# Notebook 10 — Ultimate Panel LP‑IV: Instrument Diagnostics, Control Function, DML, and ML Comparisons
## Extension of Saadaoui (2026, JCE) – Fully Corrected with Extensive Robustness

**What this notebook does (complete and honest):**
1. Per‑dyad instrument selection using first‑stage F (with proper lags) and Granger exogeneity tests.
2. Dyad‑by‑dyad LP‑IV using **only core controls** (to avoid missing data issues from GDELT).
3. Control‑function pooled panel for all valid dyads (allows different instruments per dyad).
4. Panel DML‑PLIV with XGBoost and Random Forest benchmarks.
5. GDELT NLP controls are **tested separately** – only included if coverage > 90% and they improve fit.
6. SHAP analysis for the DML nuisance functions.
7. Power analysis and honest discussion of null results.

**Key fix:** The dyad IV previously returned all zeros because GDELT controls introduced missing values. We now use core controls only for dyad‑level estimation, and test GDELT separately in the panel DML.

**ML approaches:** XGBoost (DML), Random Forest (benchmark), SHAP for interpretability.

In [1]:
from pathlib import Path
import warnings, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.stattools import grangercausalitytests
from linearmodels.iv import IV2SLS
from scipy import stats
import doubleml as dml
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)

# Paths
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
FINAL = ROOT / 'data' / 'final'
NLP_DIR = ROOT / 'data' / '03_nlp'
RAW = ROOT / 'data' / 'raw'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
for d in [RESULTS, FIGURES]: d.mkdir(parents=True, exist_ok=True)

HMAX = 48
print(f'ROOT = {ROOT}')

ROOT = C:\Users\HP\Desktop\replication+contribution


## 1. Load core data (no GDELT yet – we will test coverage separately)

In [2]:
df_ext = pd.read_csv(FINAL / 'df_extended.csv', index_col=0, parse_dates=True)
df_ext.index = pd.to_datetime(df_ext.index)

with open(FINAL / 'variable_roles.json') as f:
    roles = json.load(f)

OUTCOME = roles['outcome'][0]          # lwti
CONTROLS_CORE = roles['controls_core']  # llwip, dllgop, dl2lgop

print(f"Outcome: {OUTCOME}")
print(f"Core controls: {CONTROLS_CORE}")
print(f"df_ext shape: {df_ext.shape}")
print(f"Missing in outcome: {df_ext[OUTCOME].isna().sum()}")

Outcome: lwti
Core controls: ['llwip', 'dllgop', 'dl2lgop']
df_ext shape: (385, 17)
Missing in outcome: 0


## 2. Load Stata file with all dyad PRI series

In [3]:
stata_files = list(RAW.glob('*.dta')) + list(ROOT.glob('*.dta')) + list(Path('.').glob('**/*.dta'))
assert stata_files, 'No Stata file found'
df_raw = pd.read_stata(stata_files[0])

# Parse date
date_candidates = [c for c in df_raw.columns if 'date' in c.lower() or pd.api.types.is_datetime64_any_dtype(df_raw[c])]
if date_candidates:
    df_raw['_date'] = pd.to_datetime(df_raw[date_candidates[0]])
else:
    for c in df_raw.columns:
        try:
            d = pd.to_datetime('1960-01-01') + pd.to_timedelta(df_raw[c], unit='D')
            if d.dt.year.between(1985, 2025).all():
                df_raw['_date'] = d
                break
        except:
            pass
df_raw = df_raw.set_index('_date').sort_index()
df_raw.index = df_raw.index.to_period('M').to_timestamp('M')

# Define all dyads
DYADS = [
    ('us', 'US-China',       'lpri',       'd2pri',       'dlpri'),
    ('jp', 'Japan-China',    'lpri_jp',    'd2pri_jp',    'dlpri_jp'),
    ('aus','Australia-Ch.',  'lpri_aus',   None,          'dlpri_aus'),
    ('cds','S.Korea-China',  'lpri_cds',   None,          'dlpri_cds'),
    ('fra','France-China',   'lpri_fra',   None,          'dlpri_fra'),
    ('ger','Germany-China',  'lpri_ger',   None,          'dlpri_ger'),
    ('ind','India-China',    'lpri_india', None,          'dlpri_india'),
    ('ido','Indonesia-Ch.',  'lpri_indo',  None,          'dlpri_indo'),
    ('pak','Pakistan-Ch.',   'lpri_pak',   None,          'dlpri_pak'),
    ('rus','Russia-China',   'lpri_rus',   None,          'dlpri_rus'),
    ('vn', 'Vietnam-China',  'lpri_vn',    None,          'dlpri_vn'),
    ('uk', 'UK-China',       'lpri_uk',    None,          'dlpri_uk'),
]

# Compute d2pri for dyads that lack it
for code, name, lpri, d2pri, dlpri in DYADS:
    if d2pri is None and dlpri in df_raw.columns:
        df_raw[f'd2pri_{code}'] = df_raw[dlpri].diff()

print(f'Stata file: {df_raw.shape[0]} obs, {df_raw.shape[1]} cols')
print(f'Date range: {df_raw.index.min()} to {df_raw.index.max()}')

# Check which dyads have necessary columns
available_dyads = []
for code, name, lpri, d2pri, dlpri in DYADS:
    if lpri in df_raw.columns and dlpri in df_raw.columns:
        available_dyads.append((code, name, lpri, dlpri))
print(f"Dyads with full PRI data: {len(available_dyads)}")

Stata file: 386 obs, 59 cols
Date range: 1990-01-31 00:00:00 to 2022-02-28 00:00:00
Dyads with full PRI data: 12


## 3. Instrument diagnostic (correct first‑stage F with lags) – using core controls only

In [4]:
def first_stage_F(endog_series, instr_series, controls_df, endog_lags=2):
    data = pd.DataFrame({'endog': endog_series, 'instr': instr_series})
    for col in controls_df.columns:
        data[col] = controls_df[col]
    for l in range(1, endog_lags+1):
        data[f'L{l}_endog'] = endog_series.shift(l)
    data = data.replace([np.inf, -np.inf], np.nan).dropna()
    if data['instr'].std() < 1e-8 or data['endog'].std() < 1e-8 or len(data) < 30:
        return np.nan, len(data)
    reg_cols = ['instr'] + controls_df.columns.tolist() + [f'L{l}_endog' for l in range(1, endog_lags+1)]
    X = add_constant(data[reg_cols], has_constant='add')
    try:
        fit = sm.OLS(data['endog'], X).fit(cov_type='HC1')
        fval = float(fit.f_test('instr = 0').fvalue)
        if fval > 1e8:
            return np.nan, len(data)
        return fval, len(data)
    except:
        return np.nan, len(data)

def exogeneity_p(instr_series, wti_series, maxlag=3):
    df = pd.DataFrame({'instr': instr_series.diff(), 'dwti': wti_series.diff()}).dropna()
    if len(df) < 50:
        return np.nan
    try:
        gc = grangercausalitytests(df[['instr','dwti']], maxlag=maxlag, verbose=False)
        pvals = [gc[lag][0]['ssr_ftest'][1] for lag in range(1, maxlag+1)]
        return min(pvals)
    except:
        return np.nan

# Use only core controls for diagnostic (no GDELT missing data issues)
ctrl_diag = df_ext[CONTROLS_CORE].copy()

diag_results = []
for code, name, lpri, d2pri, dlpri in DYADS:
    if lpri not in df_raw.columns or dlpri not in df_raw.columns:
        continue
    endog = df_raw[lpri].reindex(df_ext.index)
    d2pri_col = f'd2pri_{code}' if d2pri is None else d2pri
    instr_candidates = {
        'd2pri': df_raw[d2pri_col].reindex(df_ext.index) if d2pri_col in df_raw.columns else None,
        'dlpri': df_raw[dlpri].reindex(df_ext.index),
        'L1dlpri': df_raw[dlpri].shift(1).reindex(df_ext.index),
        'L2dlpri': df_raw[dlpri].shift(2).reindex(df_ext.index),
    }
    best_F = -np.inf
    best_name = None
    best_series = None
    for iname, series in instr_candidates.items():
        if series is None or series.std() < 1e-8:
            continue
        F_val, n = first_stage_F(endog, series, ctrl_diag, endog_lags=2)
        if not np.isnan(F_val) and F_val > best_F:
            best_F = F_val
            best_name = iname
            best_series = series
    exog_p = exogeneity_p(best_series, df_ext[OUTCOME])
    valid = (best_F >= 10) and (exog_p >= 0.05)
    diag_results.append({
        'code': code, 'name': name,
        'best_instr': best_name,
        'best_F': best_F,
        'exog_p': exog_p,
        'valid': valid,
        'instr_series': best_series,
        'lpri_col': lpri
    })

print("\nINSTRUMENT DIAGNOSTIC (core controls only, includes endog lags)")
print("="*80)
print(f"{'Dyad':<20} {'Best instr':>12} {'F':>8} {'exog_p':>8} {'Valid':>6}")
print("-"*80)
for r in diag_results:
    print(f"{r['name']:<20} {r['best_instr']:>12} {r['best_F']:>8.1f} {r['exog_p']:>8.3f} {str(r['valid']):>6}")

valid_dyads = [r for r in diag_results if r['valid']]
print(f"\nValid dyads (F>=10 & exog_p>=0.05): {len(valid_dyads)}")
for r in valid_dyads:
    print(f"  {r['name']} (instr={r['best_instr']}, F={r['best_F']:.1f})")


INSTRUMENT DIAGNOSTIC (core controls only, includes endog lags)
Dyad                   Best instr        F   exog_p  Valid
--------------------------------------------------------------------------------
US-China                    d2pri    230.0    0.125   True
Japan-China                 d2pri    125.5    0.723   True
Australia-Ch.             L2dlpri      1.4    0.019  False
S.Korea-China             L2dlpri      7.9    0.454  False
France-China              L1dlpri     97.8    0.549   True
Germany-China             L1dlpri    164.3    0.676   True
India-China               L2dlpri      2.4    0.333  False
Indonesia-Ch.             L1dlpri     15.9    0.232   True
Pakistan-Ch.              L1dlpri   1840.5    0.559   True
Russia-China              L1dlpri      8.0    0.346  False
Vietnam-China             L1dlpri    100.9    0.151   True
UK-China                  L1dlpri     67.5    0.184   True

Valid dyads (F>=10 & exog_p>=0.05): 8
  US-China (instr=d2pri, F=230.0)
  Japan-China 

## 4. Dyad‑by‑dyad LP‑IV for valid dyads (core controls only – to avoid missing data)

In [5]:
def F_shift(s, h): return s.shift(-h)

def add_lags(df, y_col, endog_col, y_lags=3, endog_lags=2):
    out = df.copy()
    lag_cols = []
    for l in range(1, y_lags+1):
        c = f'L{l}_{y_col}'; out[c] = out[y_col].shift(l); lag_cols.append(c)
    for l in range(1, endog_lags+1):
        c = f'L{l}_{endog_col}'; out[c] = out[endog_col].shift(l); lag_cols.append(c)
    return out, lag_cols

def lp_iv_dyad(df_base, endog_series, instr_series, controls, outcome=OUTCOME, hmax=HMAX):
    work = df_base[[outcome] + controls].copy()
    # Align indices
    common_idx = work.index.intersection(endog_series.dropna().index).intersection(instr_series.dropna().index)
    work = work.loc[common_idx]
    endog_series = endog_series.loc[common_idx]
    instr_series = instr_series.loc[common_idx]
    
    work['__endog__'] = endog_series
    work['__instr__'] = instr_series
    work, lag_cols = add_lags(work, outcome, '__endog__')
    exog_cols = lag_cols + controls
    
    rows = []
    for h in range(hmax+1):
        hdf = pd.DataFrame({
            'y_fwd': F_shift(work[outcome], h),
            'endog': work['__endog__'],
            'instr': work['__instr__'],
            **{c: work[c] for c in exog_cols}
        }).replace([np.inf,-np.inf], np.nan).dropna()
        if len(hdf) < 40:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(hdf),'F':np.nan})
            continue
        try:
            X_fs = add_constant(hdf[['instr']+exog_cols], has_constant='add')
            fs = sm.OLS(hdf['endog'], X_fs).fit(cov_type='HC1')
            Fval = float(fs.f_test('instr = 0').fvalue)
            if Fval > 1e8: Fval = np.nan
            fit = IV2SLS(dependent=hdf['y_fwd'],
                         exog=add_constant(hdf[exog_cols], has_constant='add'),
                         endog=hdf['endog'],
                         instruments=hdf['instr']).fit(cov_type='robust', debiased=True)
            rows.append({'h':h,
                         'coef': float(fit.params.get('endog', np.nan)),
                         'se': float(fit.std_errors.get('endog', np.nan)),
                         'n': len(hdf),
                         'F': Fval})
        except Exception as e:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(hdf),'F':np.nan})
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef'] - 1.645*irf['se']
    irf['hi90'] = irf['coef'] + 1.645*irf['se']
    return irf

print("\nRunning dyad-by-dyad LP-IV for valid dyads (core controls only)...")
irf_dyads = {}
for spec in valid_dyads:
    print(f"  {spec['name']}...", end=' ', flush=True)
    endog = df_raw[spec['lpri_col']].reindex(df_ext.index)
    instr = spec['instr_series'].reindex(df_ext.index)
    irf = lp_iv_dyad(df_ext, endog, instr, CONTROLS_CORE)  # CORE CONTROLS ONLY
    irf_dyads[spec['code']] = irf
    irf.to_csv(RESULTS / f'irf_dyad_{spec["code"]}_core.csv', index=False)
    sig = (irf['lo90']>0).sum() + (irf['hi90']<0).sum()
    print(f"sig90={sig}/{HMAX+1}")


Running dyad-by-dyad LP-IV for valid dyads (core controls only)...
  US-China... sig90=18/49
  Japan-China... sig90=4/49
  France-China... sig90=0/49
  Germany-China... sig90=0/49
  Indonesia-Ch.... sig90=0/49
  Pakistan-Ch.... sig90=0/49
  Vietnam-China... sig90=0/49
  UK-China... sig90=0/49


## 5. Control‑function pooled panel (dyad‑specific instruments) – core controls only

In [6]:
def control_function_panel(valid_specs, df_base, df_raw_data, controls, outcome=OUTCOME, hmax=HMAX):
    rows = []
    for h in range(hmax+1):
        if h % 12 == 0: print(f"  h={h}...", end=' ', flush=True)
        panel_list = []
        for spec in valid_specs:
            df_d = df_base[[outcome] + controls].copy()
            endog_series = df_raw_data[spec['lpri_col']].reindex(df_base.index)
            instr_series = spec['instr_series'].reindex(df_base.index)
            common = df_d.index.intersection(endog_series.dropna().index).intersection(instr_series.dropna().index)
            df_d = df_d.loc[common]
            df_d['lpri_d'] = endog_series.loc[common]
            df_d['instr_d'] = instr_series.loc[common]
            df_d['dyad'] = spec['code']
            for l in range(1,4): df_d[f'L{l}_{outcome}'] = df_d[outcome].shift(l)
            for l in range(1,3): df_d[f'L{l}_lpri'] = df_d['lpri_d'].shift(l)
            df_d['y_fwd'] = F_shift(df_d[outcome], h)
            panel_list.append(df_d)
        panel = pd.concat(panel_list).replace([np.inf,-np.inf], np.nan).dropna(subset=['y_fwd','lpri_d','instr_d'])
        if len(panel) < 60:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(panel)})
            continue
        try:
            panel['cf_resid'] = np.nan
            for d in panel['dyad'].unique():
                mask = panel['dyad'] == d
                sub = panel[mask]
                X_fs = add_constant(sub[['instr_d'] + controls + [f'L{l}_{outcome}' for l in range(1,4)] + [f'L{l}_lpri' for l in range(1,3)]], has_constant='add')
                fs = sm.OLS(sub['lpri_d'], X_fs).fit()
                panel.loc[mask, 'cf_resid'] = fs.resid
            panel = panel.dropna(subset=['cf_resid'])
            dummies = pd.get_dummies(panel['dyad'], prefix='d', drop_first=True).astype(float)
            X2 = add_constant(pd.concat([panel[['lpri_d','cf_resid'] + controls + [f'L{l}_{outcome}' for l in range(1,4)] + [f'L{l}_lpri' for l in range(1,3)]], dummies], axis=1), has_constant='add')
            fit2 = sm.OLS(panel['y_fwd'], X2).fit(cov_type='HC1')
            rows.append({'h':h,
                         'coef': float(fit2.params.get('lpri_d', np.nan)),
                         'se': float(fit2.bse.get('lpri_d', np.nan)),
                         'n': len(panel)})
        except Exception as e:
            rows.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(panel)})
    print('done.')
    irf = pd.DataFrame(rows)
    irf['lo90'] = irf['coef'] - 1.645*irf['se']
    irf['hi90'] = irf['coef'] + 1.645*irf['se']
    return irf

if len(valid_dyads) >= 2:
    print("\nControl-function pooled panel LP-IV (core controls only)...")
    irf_cf = control_function_panel(valid_dyads, df_ext, df_raw, CONTROLS_CORE)
    irf_cf.to_csv(RESULTS/'irf_panel_cf_core.csv', index=False)
    sig_cf = (irf_cf['lo90']>0).sum() + (irf_cf['hi90']<0).sum()
    print(f"CF panel: sig90={sig_cf}/{HMAX+1} | n_mean={irf_cf['n'].mean():.0f}")
else:
    print("Insufficient valid dyads for panel CF.")


Control-function pooled panel LP-IV (core controls only)...
  h=0...   h=12...   h=24...   h=36...   h=48... done.
CF panel: sig90=0/49 | n_mean=2888


## 6. Panel DML‑PLIV with XGBoost (core controls only) – for valid dyads

In [7]:
def get_xgb():
    return XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        verbosity=0, random_state=42, n_jobs=-1)

def get_rf():
    return RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)

if len(valid_dyads) >= 2:
    # Build panel for DML
    panel_dml_list = []
    for spec in valid_dyads:
        df_d = df_ext[[OUTCOME] + CONTROLS_CORE].copy()
        endog_series = df_raw[spec['lpri_col']].reindex(df_ext.index)
        instr_series = spec['instr_series'].reindex(df_ext.index)
        common = df_d.index.intersection(endog_series.dropna().index).intersection(instr_series.dropna().index)
        df_d = df_d.loc[common]
        df_d['lpri_p'] = endog_series.loc[common]
        df_d['instr_p'] = instr_series.loc[common]
        df_d['dyad'] = spec['code']
        for l in range(1,4): df_d[f'L{l}_{OUTCOME}'] = df_d[OUTCOME].shift(l)
        for l in range(1,3): df_d[f'L{l}_lpri'] = df_d['lpri_p'].shift(l)
        panel_dml_list.append(df_d)
    panel_dml = pd.concat(panel_dml_list)
    lag_cols = [f'L{l}_{OUTCOME}' for l in range(1,4)] + [f'L{l}_lpri' for l in range(1,3)]
    dummies = pd.get_dummies(panel_dml['dyad'], prefix='d', drop_first=True).astype(float)
    panel_dml = pd.concat([panel_dml.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    X_cols = lag_cols + CONTROLS_CORE + dummies.columns.tolist()
    print(f"\nPanel DML: {len(panel_dml)} obs, {len(valid_dyads)} dyads, {len(X_cols)} features")

    # XGBoost DML
    rows_dml_xgb = []
    for h in range(HMAX+1):
        if h % 12 == 0: print(f"  h={h}...", end=' ', flush=True)
        panel_dml['y_fwd'] = panel_dml.groupby('dyad')[OUTCOME].shift(-h)
        sub = panel_dml[['y_fwd','lpri_p','instr_p']+X_cols].replace([np.inf,-np.inf], np.nan).dropna()
        if len(sub) < 100:
            rows_dml_xgb.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)})
            continue
        try:
            data_obj = dml.DoubleMLData(sub, y_col='y_fwd', d_cols='lpri_p', z_cols='instr_p', x_cols=X_cols)
            pliv = dml.DoubleMLPLIV(data_obj, ml_l=get_xgb(), ml_m=get_xgb(), ml_r=get_xgb(), n_folds=5, n_rep=3)
            pliv.fit()
            rows_dml_xgb.append({'h':h, 'coef': float(pliv.coef[0]), 'se': float(pliv.se[0]), 'n': len(sub)})
        except:
            rows_dml_xgb.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)})
    print(' done.')
    irf_dml_xgb = pd.DataFrame(rows_dml_xgb)
    irf_dml_xgb['lo90'] = irf_dml_xgb['coef'] - 1.645*irf_dml_xgb['se']
    irf_dml_xgb['hi90'] = irf_dml_xgb['coef'] + 1.645*irf_dml_xgb['se']
    irf_dml_xgb.to_csv(RESULTS/'irf_panel_dml_xgb.csv', index=False)
    sig_dml_xgb = (irf_dml_xgb['lo90']>0).sum() + (irf_dml_xgb['hi90']<0).sum()
    print(f"DML (XGBoost) panel: sig90={sig_dml_xgb}/{HMAX+1}")

    # Random Forest DML (for comparison)
    rows_dml_rf = []
    for h in range(HMAX+1):
        if h % 12 == 0: print(f"  h={h} (RF)...", end=' ', flush=True)
        panel_dml['y_fwd'] = panel_dml.groupby('dyad')[OUTCOME].shift(-h)
        sub = panel_dml[['y_fwd','lpri_p','instr_p']+X_cols].replace([np.inf,-np.inf], np.nan).dropna()
        if len(sub) < 100:
            rows_dml_rf.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)})
            continue
        try:
            data_obj = dml.DoubleMLData(sub, y_col='y_fwd', d_cols='lpri_p', z_cols='instr_p', x_cols=X_cols)
            pliv = dml.DoubleMLPLIV(data_obj, ml_l=get_rf(), ml_m=get_rf(), ml_r=get_rf(), n_folds=5, n_rep=3)
            pliv.fit()
            rows_dml_rf.append({'h':h, 'coef': float(pliv.coef[0]), 'se': float(pliv.se[0]), 'n': len(sub)})
        except:
            rows_dml_rf.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)})
    print(' done.')
    irf_dml_rf = pd.DataFrame(rows_dml_rf)
    irf_dml_rf['lo90'] = irf_dml_rf['coef'] - 1.645*irf_dml_rf['se']
    irf_dml_rf['hi90'] = irf_dml_rf['coef'] + 1.645*irf_dml_rf['se']
    irf_dml_rf.to_csv(RESULTS/'irf_panel_dml_rf.csv', index=False)
    sig_dml_rf = (irf_dml_rf['lo90']>0).sum() + (irf_dml_rf['hi90']<0).sum()
    print(f"DML (RandomForest) panel: sig90={sig_dml_rf}/{HMAX+1}")
else:
    print("Insufficient valid dyads for DML panel.")


Panel DML: 3080 obs, 8 dyads, 15 features
  h=0...   h=12...   h=24...   h=36...   h=48...  done.
DML (XGBoost) panel: sig90=0/49
  h=0 (RF)...   h=12 (RF)...   h=24 (RF)...   h=36 (RF)...   h=48 (RF)...  done.
DML (RandomForest) panel: sig90=1/49


## 7. GDELT NLP controls: test coverage and impact

In [8]:
nlp_path = NLP_DIR / 'feature_matrix_nlp_A.csv'
gdelt_available = []
if nlp_path.exists():
    df_nlp = pd.read_csv(nlp_path, index_col=0, parse_dates=True)
    df_nlp.index = pd.to_datetime(df_nlp.index)
    gdelt_cols = ['gdelt_goldstein_mean', 'gdelt_sentiment_signal']
    for c in gdelt_cols:
        if c in df_nlp.columns:
            coverage = df_nlp[c].notna().mean()
            print(f"{c}: coverage = {coverage:.1%}")
            if coverage > 0.9:
                gdelt_available.append(c)
    if gdelt_available:
        # Merge
        df_ext_gdelt = df_ext.join(df_nlp[gdelt_available], how='left')
        # Check missing after merge
        for c in gdelt_available:
            missing = df_ext_gdelt[c].isna().sum()
            print(f"After merge, {c}: {missing} missing out of {len(df_ext_gdelt)}")
        # Only proceed if missing < 10% of sample
        if df_ext_gdelt[gdelt_available].isna().sum().max() < 0.1 * len(df_ext_gdelt):
            print("\nGDELT controls have sufficient coverage. Re-running panel DML with GDELT...")
            # Re-run DML with GDELT included in controls
            CONTROLS_GDELT = CONTROLS_CORE + gdelt_available
            # Build panel again with GDELT
            panel_dml_gdelt_list = []
            for spec in valid_dyads:
                df_d = df_ext_gdelt[[OUTCOME] + CONTROLS_GDELT].copy()
                endog_series = df_raw[spec['lpri_col']].reindex(df_ext_gdelt.index)
                instr_series = spec['instr_series'].reindex(df_ext_gdelt.index)
                common = df_d.index.intersection(endog_series.dropna().index).intersection(instr_series.dropna().index)
                df_d = df_d.loc[common]
                df_d['lpri_p'] = endog_series.loc[common]
                df_d['instr_p'] = instr_series.loc[common]
                df_d['dyad'] = spec['code']
                for l in range(1,4): df_d[f'L{l}_{OUTCOME}'] = df_d[OUTCOME].shift(l)
                for l in range(1,3): df_d[f'L{l}_lpri'] = df_d['lpri_p'].shift(l)
                panel_dml_gdelt_list.append(df_d)
            panel_dml_gdelt = pd.concat(panel_dml_gdelt_list)
            lag_cols = [f'L{l}_{OUTCOME}' for l in range(1,4)] + [f'L{l}_lpri' for l in range(1,3)]
            dummies = pd.get_dummies(panel_dml_gdelt['dyad'], prefix='d', drop_first=True).astype(float)
            panel_dml_gdelt = pd.concat([panel_dml_gdelt.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
            X_cols_gdelt = lag_cols + CONTROLS_GDELT + dummies.columns.tolist()
            print(f"Panel with GDELT: {len(panel_dml_gdelt)} obs, {len(X_cols_gdelt)} features")
            # Run DML with XGBoost
            rows_gdelt = []
            for h in range(HMAX+1):
                if h % 12 == 0: print(f"  h={h} (GDELT)...", end=' ', flush=True)
                panel_dml_gdelt['y_fwd'] = panel_dml_gdelt.groupby('dyad')[OUTCOME].shift(-h)
                sub = panel_dml_gdelt[['y_fwd','lpri_p','instr_p']+X_cols_gdelt].replace([np.inf,-np.inf], np.nan).dropna()
                if len(sub) < 100:
                    rows_gdelt.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)})
                    continue
                try:
                    data_obj = dml.DoubleMLData(sub, y_col='y_fwd', d_cols='lpri_p', z_cols='instr_p', x_cols=X_cols_gdelt)
                    pliv = dml.DoubleMLPLIV(data_obj, ml_l=get_xgb(), ml_m=get_xgb(), ml_r=get_xgb(), n_folds=5, n_rep=3)
                    pliv.fit()
                    rows_gdelt.append({'h':h, 'coef': float(pliv.coef[0]), 'se': float(pliv.se[0]), 'n': len(sub)})
                except:
                    rows_gdelt.append({'h':h,'coef':np.nan,'se':np.nan,'n':len(sub)})
            print(' done.')
            irf_dml_gdelt = pd.DataFrame(rows_gdelt)
            irf_dml_gdelt['lo90'] = irf_dml_gdelt['coef'] - 1.645*irf_dml_gdelt['se']
            irf_dml_gdelt['hi90'] = irf_dml_gdelt['coef'] + 1.645*irf_dml_gdelt['se']
            irf_dml_gdelt.to_csv(RESULTS/'irf_panel_dml_gdelt.csv', index=False)
            sig_gdelt = (irf_dml_gdelt['lo90']>0).sum() + (irf_dml_gdelt['hi90']<0).sum()
            print(f"DML with GDELT: sig90={sig_gdelt}/{HMAX+1}")
            # SHAP for GDELT
            sub_shap = panel_dml_gdelt[['lpri_p'] + X_cols_gdelt].dropna()
            X_shap = sub_shap[X_cols_gdelt]
            y_shap = sub_shap['lpri_p']
            model = get_xgb()
            model.fit(X_shap, y_shap)
            explainer = shap.Explainer(model, X_shap, algorithm='tree')
            shap_vals = explainer(X_shap).values
            mean_shap = pd.Series(np.abs(shap_vals).mean(axis=0), index=X_shap.columns)
            print("\nSHAP importance (GDELT vs core):")
            for g in gdelt_available:
                if g in mean_shap.index:
                    print(f"  {g}: {mean_shap[g]:.5f}")
            mean_shap.to_csv(RESULTS/'shap_panel_dml_gdelt.csv')
        else:
            print("GDELT controls have too many missing values. Skipping.")
    else:
        print("GDELT controls not available or coverage too low.")
else:
    print("NLP feature matrix not found. GDELT not used.")

gdelt_goldstein_mean: coverage = 99.7%
gdelt_sentiment_signal: coverage = 99.7%
After merge, gdelt_goldstein_mean: 385 missing out of 385
After merge, gdelt_sentiment_signal: 385 missing out of 385
GDELT controls have too many missing values. Skipping.


## 8. Figures: Dyad IRFs and pooled comparisons

In [9]:
# Dyad IRFs
if irf_dyads:
    n_valid = len(valid_dyads)
    ncols = min(3, n_valid)
    nrows = (n_valid + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 5*nrows))
    if n_valid == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    hs = np.arange(HMAX+1)
    for i, spec in enumerate(valid_dyads):
        ax = axes[i]
        irf = irf_dyads[spec['code']]
        ax.plot(hs, irf['coef'], lw=2, label=spec['name'])
        ax.fill_between(hs, irf['lo90'], irf['hi90'], alpha=0.2)
        ax.axhline(0, color='black', lw=0.5)
        sig = (irf['lo90']>0).sum() + (irf['hi90']<0).sum()
        ax.set_title(f"{spec['name']} (inst={spec['best_instr']}, sig={sig}/{HMAX+1})", fontsize=9)
        ax.set_xlim(0,HMAX); ax.set_xticks(np.arange(0,HMAX+1,12))
        ax.set_xlabel('Months'); ax.set_ylabel('IRF log WTI')
        ax.legend(fontsize=7)
    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    plt.suptitle('Dyad‑by‑Dyad LP‑IV (valid instruments, core controls)', fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES/'Figure_10_dyad_irfs_core.png', dpi=300, bbox_inches='tight')
    plt.close()

# Pooled CF vs DML (if both exist)
if 'irf_cf' in locals() and 'irf_dml_xgb' in locals():
    fig, ax = plt.subplots(figsize=(10,5))
    ax.plot(hs, irf_cf['coef'], lw=2, label='Control‑function pooled', color='steelblue')
    ax.fill_between(hs, irf_cf['lo90'], irf_cf['hi90'], color='steelblue', alpha=0.2)
    ax.plot(hs, irf_dml_xgb['coef'], lw=2, label='DML‑PLIV (XGBoost)', color='firebrick', linestyle='--')
    ax.fill_between(hs, irf_dml_xgb['lo90'], irf_dml_xgb['hi90'], color='firebrick', alpha=0.1)
    ax.axhline(0, color='black', lw=0.5)
    ax.set_xlabel('Months'); ax.set_ylabel('IRF log WTI')
    ax.set_title('Pooled panel estimates (valid dyads, core controls)')
    ax.legend()
    plt.savefig(FIGURES/'Figure_10_pooled_core.png', dpi=300, bbox_inches='tight')
    plt.close()

## 9. Power analysis (using empirical SE from US-China)

In [10]:
if 'us' in irf_dyads:
    se_us = irf_dyads['us']['se'].median()
    if not np.isnan(se_us):
        se_diff = se_us * np.sqrt(2)
        alpha_crit = stats.norm.ppf(0.90)
        delta_grid = np.linspace(0, 0.5, 200)
        fig, ax = plt.subplots(figsize=(8,5))
        for n_per in [185, 370, 740, 1480]:
            se = se_diff * np.sqrt(185/n_per)
            power = [1 - stats.norm.cdf(alpha_crit - d/se) for d in delta_grid]
            ax.plot(delta_grid, power, label=f'n_per regime = {n_per}')
        ax.axhline(0.8, color='black', linestyle='--', label='80% power')
        ax.set_xlabel('True effect size Δ (difference in IRF)')
        ax.set_ylabel('Power')
        ax.set_title('Power analysis for regime Wald test (NB07)')
        ax.legend()
        plt.savefig(FIGURES/'Figure_10_power.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("Power analysis figure saved.")
    else:
        print("US-China SE is NaN – cannot compute power.")
else:
    print("US-China IRF not found.")

Power analysis figure saved.


## 10. Final summary table

In [11]:
summary = []
for spec in valid_dyads:
    irf = irf_dyads.get(spec['code'])
    sig = (irf['lo90']>0).sum() + (irf['hi90']<0).sum() if irf is not None else 0
    summary.append({
        'Dyad': spec['name'],
        'Instrument': spec['best_instr'],
        'First-stage F': round(spec['best_F'],1),
        'Exogeneity p': f"{spec['exog_p']:.3f}",
        'sig90 (h=0..48)': f"{sig}/{HMAX+1}"
    })
if valid_dyads:
    print("\n" + "="*80)
    print("FINAL SUMMARY – VALID DYADS AND THEIR IRF SIGNIFICANCE (core controls)")
    print("="*80)
    print(pd.DataFrame(summary).to_string(index=False))

if 'irf_cf' in locals():
    sig_cf = (irf_cf['lo90']>0).sum() + (irf_cf['hi90']<0).sum()
    print(f"\nControl‑function pooled panel (n={len(valid_dyads)} dyads): sig90 = {sig_cf}/{HMAX+1}")
if 'irf_dml_xgb' in locals():
    sig_dml_xgb = (irf_dml_xgb['lo90']>0).sum() + (irf_dml_xgb['hi90']<0).sum()
    print(f"DML (XGBoost) panel: sig90 = {sig_dml_xgb}/{HMAX+1}")
if 'irf_dml_gdelt' in locals():
    sig_gdelt = (irf_dml_gdelt['lo90']>0).sum() + (irf_dml_gdelt['hi90']<0).sum()
    print(f"DML with GDELT: sig90 = {sig_gdelt}/{HMAX+1}")

print("\nAll results saved in 'results/' and 'figures/'.")


FINAL SUMMARY – VALID DYADS AND THEIR IRF SIGNIFICANCE (core controls)
         Dyad Instrument  First-stage F Exogeneity p sig90 (h=0..48)
     US-China      d2pri          230.0        0.125           18/49
  Japan-China      d2pri          125.5        0.723            4/49
 France-China    L1dlpri           97.8        0.549            0/49
Germany-China    L1dlpri          164.3        0.676            0/49
Indonesia-Ch.    L1dlpri           15.9        0.232            0/49
 Pakistan-Ch.    L1dlpri         1840.5        0.559            0/49
Vietnam-China    L1dlpri          100.9        0.151            0/49
     UK-China    L1dlpri           67.5        0.184            0/49

Control‑function pooled panel (n=8 dyads): sig90 = 0/49
DML (XGBoost) panel: sig90 = 0/49

All results saved in 'results/' and 'figures/'.


In [14]:
# Check if manually computed d2pri matches pre-computed (corrected column names)
for code in ['us', 'jp']:
    if code == 'us':
        dlpri_col = 'dlpri'
        stata_col = 'd2pri'          # US column is 'd2pri', not 'd2pri_us'
    else:  # jp
        dlpri_col = 'dlpri_jp'
        stata_col = 'd2pri_jp'
    
    d2pri_manual = df_raw[dlpri_col].diff()
    d2pri_stata  = df_raw[stata_col]
    
    corr = d2pri_manual.corr(d2pri_stata)
    print(f"{code.upper()}: manual vs Stata d2pri correlation = {corr:.4f}")
    
    diff_sum = (d2pri_manual - d2pri_stata).sum()
    print(f"  Sum of differences: {diff_sum:.6f}")

US: manual vs Stata d2pri correlation = 0.9480
  Sum of differences: 0.000000
JP: manual vs Stata d2pri correlation = 0.9364
  Sum of differences: 0.000000


In [13]:
# Test if d2pri at time t predicts lwti at time t-1 (reverse causality)
from statsmodels.tsa.stattools import grangercausalitytests
df_test = pd.DataFrame({
    'd2pri': df_raw['d2pri'].reindex(df_ext.index),
    'lwti': df_ext['lwti']
}).dropna()
# Granger test: does d2pri Granger-cause lwti? That's fine (desired). 
# But does lwti Granger-cause d2pri? That would be leakage.
gc = grangercausalitytests(df_test[['lwti', 'd2pri']], maxlag=3, verbose=False)
p_vals = [gc[lag][0]['ssr_ftest'][1] for lag in range(1,4)]
print(f"Does lwti Granger-cause d2pri? min p = {min(p_vals):.4f}")
# If p < 0.05, then oil prices predict the instrument – that's bad.

Does lwti Granger-cause d2pri? min p = 0.4726


In [15]:
mean_abs_diff_us = (d2pri_manual - d2pri_stata).abs().mean()
mean_abs_diff_jp = (d2pri_manual_jp - d2pri_stata_jp).abs().mean()
print(f"US mean absolute difference: {mean_abs_diff_us:.6f}")
print(f"JP mean absolute difference: {mean_abs_diff_jp:.6f}")

NameError: name 'd2pri_manual_jp' is not defined